# GhostID v3 — LSTM Encoder + ArcFace

Train a 128-dim behavioral embedding encoder on CMU DSL Keystroke Dynamics.

**Dataset:** `DSL-StrongPasswordData.csv` — 51 users, 20,400 sessions

**Outputs:** `ghostid_encoder.onnx`, `scaler_params.json`, analysis plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_curve, auc
import json

df = pd.read_csv('/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv')
print(f'Shape: {df.shape}, Users: {df["subject"].nunique()}')

In [ ]:
feature_cols = [c for c in df.columns if c not in ['subject', 'sessionIndex', 'rep']]
H_cols = [c for c in feature_cols if c.startswith('H.')]
for i in range(len(H_cols) - 1):
    df[f'ratio_{i}'] = df[H_cols[i]] / (df[H_cols[i + 1]] + 1e-8)
ratio_cols = [c for c in df.columns if c.startswith('ratio_')]
all_features = feature_cols + ratio_cols
print(f'Total features: {len(all_features)}')

scaler = StandardScaler()
X = scaler.fit_transform(df[all_features].values).astype(np.float32)
with open('scaler_params.json', 'w') as f:
    json.dump({'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()}, f)

In [ ]:
class GhostIDEncoder(nn.Module):
    def __init__(self, input_size=41, hidden_size=256, num_layers=2, embed_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.fc = nn.Linear(hidden_size, embed_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.layer_norm(out[:, -1, :])
        out = self.dropout(out)
        out = self.fc(out)
        return nn.functional.normalize(out, p=2, dim=1)

class ArcFaceLoss(nn.Module):
    def __init__(self, embed_dim=128, num_classes=51, s=64.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embed_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = np.cos(m)
        self.sin_m = np.sin(m)
        self.th = np.cos(np.pi - m)
        self.mm = np.sin(np.pi - m) * m

    def forward(self, embeddings, labels):
        cosine = nn.functional.linear(
            embeddings,
            nn.functional.normalize(self.weight)
        )
        cosine = cosine.clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt(1.0 - cosine ** 2)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return nn.CrossEntropyLoss()(output, labels)

print('Model classes defined')

In [ ]:
train_idx = []
test_idx = []
subjects_array = df['subject'].values

for subject in df['subject'].unique():
    user_mask = df['subject'] == subject
    user_indices = df[user_mask].index.tolist()
    split = int(len(user_indices) * 0.8)
    train_idx.extend(user_indices[:split])
    test_idx.extend(user_indices[split:])

X_train, X_test = X[train_idx], X[test_idx]
le = LabelEncoder()
all_labels = le.fit_transform(subjects_array)
y_train, y_test = all_labels[train_idx], all_labels[test_idx]

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

EPOCHS = 60
BATCH_SIZE = 64

X_train_t = torch.FloatTensor(X_train.reshape(-1, 1, 41)).to(device)
y_train_t = torch.LongTensor(y_train).to(device)

class KeystrokeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(
    KeystrokeDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True
)

encoder = GhostIDEncoder().to(device)
arcface = ArcFaceLoss(embed_dim=128, num_classes=51).to(device)

optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(arcface.parameters()),
    lr=0.001, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

best_loss = float('inf')
train_losses = []

for epoch in range(EPOCHS):
    encoder.train()
    arcface.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        embeddings = encoder(X_batch)
        loss = arcface(embeddings, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(encoder.state_dict(), 'best_ghostid_encoder.pt')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg_loss:.4f}  Best: {best_loss:.4f}')

print(f'Training complete. Best loss: {best_loss:.4f}')

In [ ]:
encoder.load_state_dict(torch.load('best_ghostid_encoder.pt', map_location=device))
encoder.eval()

X_test_t = torch.FloatTensor(X_test.reshape(-1, 1, 41)).to(device)
subjects_test = subjects_array[test_idx]

genuine_scores = []
impostor_scores = []

with torch.no_grad():
    for subject in df['subject'].unique():
        user_mask = subjects_test == subject
        user_X = X_test_t[user_mask]
        if len(user_X) < 3:
            continue

        enroll_embs = encoder(user_X[:2])
        baseline = enroll_embs.mean(dim=0, keepdim=True)
        baseline = nn.functional.normalize(baseline, p=2, dim=1).squeeze()

        probe_genuine = encoder(user_X[2:])
        for emb in probe_genuine:
            genuine_scores.append(torch.dot(emb, baseline).item() * 100)

        other_mask = subjects_test != subject
        other_X = X_test_t[other_mask]
        perm = torch.randperm(len(other_X))[:20]
        probe_impostor = encoder(other_X[perm])
        for emb in probe_impostor:
            impostor_scores.append(torch.dot(emb, baseline).item() * 100)

genuine_scores = np.array(genuine_scores)
impostor_scores = np.array(impostor_scores)

print(f'Genuine  — mean: {genuine_scores.mean():.1f}  std: {genuine_scores.std():.1f}')
print(f'Impostor — mean: {impostor_scores.mean():.1f}  std: {impostor_scores.std():.1f}')
print(f'Gap: {genuine_scores.mean() - impostor_scores.mean():.1f} points')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#06080c')

ax1 = axes[0]
ax1.set_facecolor('#0d1117')
ax1.hist(genuine_scores, bins=40, alpha=0.7, color='#00e5a0',
         label=f'Genuine (n={len(genuine_scores)})')
ax1.hist(impostor_scores, bins=40, alpha=0.7, color='#ff2d55',
         label=f'Impostor (n={len(impostor_scores)})')
ax1.axvline(85, color='white', linestyle='--', alpha=0.6, label='85 — SILENT_PASS')
ax1.axvline(40, color='#ff6b2b', linestyle='--', alpha=0.6, label='40 — HARD_STOP')
ax1.set_xlabel('Confidence Score', color='white')
ax1.set_ylabel('Count', color='white')
ax1.set_title('Genuine vs Impostor Score Distribution', color='white')
ax1.tick_params(colors='white')
ax1.legend()

labels_combined = np.concatenate([np.ones(len(genuine_scores)), np.zeros(len(impostor_scores))])
scores_combined = np.concatenate([genuine_scores, impostor_scores])
fpr, tpr, thresholds = roc_curve(labels_combined, scores_combined)
roc_auc = auc(fpr, tpr)

ax2 = axes[1]
ax2.set_facecolor('#0d1117')
ax2.plot(fpr, tpr, color='#4a9eff', lw=2, label=f'AUC = {roc_auc:.3f}')
ax2.plot([0,1],[0,1],'w--',alpha=0.3)
ax2.set_xlabel('False Accept Rate', color='white')
ax2.set_ylabel('True Accept Rate', color='white')
ax2.set_title('ROC Curve', color='white')
ax2.tick_params(colors='white')
ax2.legend()

plt.tight_layout()
plt.savefig('ghostid_analysis.png', dpi=150, bbox_inches='tight', facecolor='#06080c')
plt.show()
print(f'AUC: {roc_auc:.4f}')

fnr = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
print(f'EER: {eer*100:.2f}%')

for name, thresh in [('HARD_STOP',40),('TYPING_CHALLENGE',69),('SOFT_NUDGE',84)]:
    far = np.mean(impostor_scores >= thresh)*100
    frr = np.mean(genuine_scores < thresh)*100
    print(f'{name:20s} (>={thresh})  FAR:{far:.1f}%  FRR:{frr:.1f}%')

In [ ]:
encoder.eval()
with torch.no_grad():
    real_user = df['subject'].unique()[0]
    imp_user = df['subject'].unique()[5]

    real_X = torch.FloatTensor(
        X[df['subject'].values == real_user].reshape(-1, 1, 41)
    ).to(device)
    imp_X = torch.FloatTensor(
        X[df['subject'].values == imp_user].reshape(-1, 1, 41)
    ).to(device)

    baseline_emb = encoder(real_X[:2]).mean(dim=0)
    baseline_emb = nn.functional.normalize(baseline_emb.unsqueeze(0), p=2, dim=1).squeeze()
    current_baseline = baseline_emb.clone()

    scores_timeline = []
    for i in range(200):
        if i < 50:
            session_emb = encoder(real_X[i:i+1]).squeeze()
        else:
            idx = (i - 50) % len(imp_X)
            session_emb = encoder(imp_X[idx:idx+1]).squeeze()

        score = torch.dot(session_emb, current_baseline).item() * 100
        scores_timeline.append(score)

        if score >= 85:
            new_b = current_baseline * 0.92 + session_emb * 0.08
            current_baseline = nn.functional.normalize(new_b.unsqueeze(0), p=2, dim=1).squeeze()

fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor('#06080c')
ax.set_facecolor('#0d1117')
ax.plot(scores_timeline, color='#4a9eff', lw=1.5)
ax.axvline(50, color='#ff2d55', linestyle='--', label='Impostor takeover')
ax.axhline(40, color='#ff2d55', alpha=0.4, linestyle=':', label='HARD_STOP (40)')
ax.axhline(85, color='#00e5a0', alpha=0.4, linestyle=':', label='SILENT_PASS (85)')
ax.fill_between(range(200), 0, 40, alpha=0.05, color='#ff2d55')
ax.set_xlabel('Session', color='white')
ax.set_ylabel('Score', color='white')
ax.set_title('EMA Baseline Poisoning Resistance', color='white')
ax.tick_params(colors='white')
ax.legend()
plt.tight_layout()
plt.savefig('poisoning_resistance.png', dpi=150, bbox_inches='tight', facecolor='#06080c')
plt.show()

In [ ]:
import os
import torch.onnx
import onnx
import onnxruntime as ort

encoder.load_state_dict(torch.load('best_ghostid_encoder.pt', map_location='cpu'))
encoder.eval()
encoder.cpu()

dummy = torch.randn(1, 1, 41)
torch.onnx.export(
    encoder, dummy, 'ghostid_encoder.onnx',
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['features'],
    output_names=['embedding'],
    dynamic_axes={'features': {0: 'batch_size'}, 'embedding': {0: 'batch_size'}}
)

model_check = onnx.load('ghostid_encoder.onnx')
onnx.checker.check_model(model_check)
sess = ort.InferenceSession('ghostid_encoder.onnx')
out = sess.run(None, {'features': np.random.randn(1, 1, 41).astype(np.float32)})
assert out[0].shape == (1, 128)
norm = np.linalg.norm(out[0])
assert 0.98 < norm < 1.02, f'Expected unit norm, got {norm:.4f}'
size_kb = os.path.getsize('ghostid_encoder.onnx') / 1024
print(f'ONNX export verified. Shape: {out[0].shape}  Norm: {norm:.4f}  Size: {size_kb:.1f}KB')

scaler_params = {
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist()
}
with open('scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)
print('scaler_params.json saved')
print('Download both files and place in backend/ml/')